In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device: CPU")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
Device: CPU


In [ ]:
!pip install -q transformers datasets
!pip install -q torch-geometric
!pip install -q librosa soundfile
!pip install -q scikit-learn pandas matplotlib seaborn networkx pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00


In [ ]:
CONFIG = {
    "dataset": "FMA-small",
    "max_tracks": 800,
    "sample_rate": 22050,
    "segment_seconds": 5,
    "max_text_length": 128,
    "bert_model": "distilbert-base-uncased",
    "batch_size": 4,
    "epochs": 3,
    "learning_rate": 2e-5,
    "num_classes": 8
}

CONFIG

{'dataset': 'FMA-small',
 'max_tracks': 800,
 'sample_rate': 22050,
 'segment_seconds': 5,
 'max_text_length': 128,
 'bert_model': 'distilbert-base-uncased',
 'batch_size': 4,
 'epochs': 3,
 'learning_rate': 2e-05,
 'num_classes': 8}

# FMA

In [ ]:
!mkdir -p /content/fma

In [ ]:
!wget -O /content/fma/fma_metadata.zip \
"https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"

--2026-09-11 10:15:53--  https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
Resolving os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)... 86.119.28.16, 2001:620:5ca1:201::214
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|86.119.28.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358412441 (342M) [application/zip]
Saving to: ‘/content/fma/fma_metadata.zip’

/content/fma/fma_me 100%[===================>] 341.81M  25.5MB/s    in 14s     

2026-09-11 10:16:08 (23.7 MB/s) - ‘/content/fma/fma_metadata.zip’ saved [358412441/358412441]



In [ ]:
!wget -O /content/fma/fma_small.zip \
"https://os.unil.cloud.switch.ch/fma/fma_small.zip"

--2026-09-11 10:16:15--  https://os.unil.cloud.switch.ch/fma/fma_small.zip
Resolving os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)... 86.119.28.16, 2001:620:5ca1:201::214
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|86.119.28.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7679594875 (7.2G) [application/zip]
Saving to: ‘/content/fma/fma_small.zip’

/content/fma/fma_sm 100%[===================>]   7.15G  26.7MB/s    in 4m 53s  

2026-09-11 10:21:09 (25.0 MB/s) - ‘/content/fma/fma_small.zip’ saved [7679594875/7679594875]



In [ ]:
!mkdir -p /content/fma/metadata
!unzip -q /content/fma/fma_metadata.zip -d /content/fma/metadata

In [ ]:
import pandas as pd

tracks = pd.read_csv(
    "/content/fma/metadata/fma_metadata/tracks.csv",
    header=[0, 1],
    index_col=0
)

print(tracks.shape)
tracks.head()

(106574, 52)


album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]

# Track/File Extraction

In [ ]:
import zipfile
import os

with zipfile.ZipFile("/content/fma/fma_small.zip", "r") as z:
    members = z.namelist()

available_ids = []

for member in members:
    if member.lower().endswith(".mp3"):
        filename = os.path.basename(member)
        try:
            track_id = int(os.path.splitext(filename)[0])
            available_ids.append(track_id)
        except ValueError:
            pass

available_ids = sorted(set(available_ids))

print("Tracks actually in FMA-small:", len(available_ids))

Tracks actually in FMA-small: 8000


In [ ]:
genre_column = ("track", "genre_top")

genres = [
    "Electronic",
    "Experimental",
    "Folk",
    "Hip-Hop",
    "Instrumental",
    "International",
    "Pop",
    "Rock"
]

available_tracks = tracks.loc[
    tracks.index.intersection(available_ids)
].copy()

selected_ids = []

for genre in genres:
    genre_ids = available_tracks[
        available_tracks[genre_column] == genre
    ].index[:100]

    selected_ids.extend(genre_ids.tolist())

print("Selected tracks:", len(selected_ids))

Selected tracks: 800


In [ ]:
import zipfile
import os
import shutil

audio_dir = "/content/fma/audio"
os.makedirs(audio_dir, exist_ok=True)

selected_ids_set = set(int(x) for x in selected_ids)

with zipfile.ZipFile("/content/fma/fma_small.zip", "r") as z:
    extracted = 0

    for member in z.namelist():

        if not member.lower().endswith(".mp3"):
            continue

        filename = os.path.basename(member)

        try:
            track_id = int(os.path.splitext(filename)[0])
        except ValueError:
            continue

        if track_id not in selected_ids_set:
            continue

        output_path = os.path.join(
            audio_dir,
            filename
        )

        if os.path.exists(output_path):
            continue

        with z.open(member) as source:
            with open(output_path, "wb") as target:
                shutil.copyfileobj(source, target)

        extracted += 1

        if extracted % 50 == 0:
            print(f"Extracted {extracted}/800")

print("Newly extracted:", extracted)

Extracted 50/800
Extracted 100/800
Extracted 150/800
Extracted 200/800
Extracted 250/800
Extracted 300/800
Extracted 350/800
Extracted 400/800
Extracted 450/800
Extracted 500/800
Extracted 550/800
Extracted 600/800
Extracted 650/800
Extracted 700/800
Extracted 750/800
Extracted 800/800
Newly extracted: 800


In [ ]:
selected_audio_count = sum(
    os.path.exists(
        os.path.join(
            audio_dir,
            f"{int(track_id):06d}.mp3"
        )
    )
    for track_id in selected_ids
)

print("Selected tracks:", len(selected_ids))
print("Selected audio files:", selected_audio_count)
print(
    "Missing:",
    len(selected_ids) - selected_audio_count
)

Selected tracks: 800
Selected audio files: 800
Missing: 0


# Train/validation/test split

In [ ]:
from sklearn.model_selection import train_test_split

ids = selected_ids

train_ids, temp_ids = train_test_split(
    ids,
    test_size=0.30,
    random_state=42,
    stratify=tracks.loc[ids, genre_column]
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=tracks.loc[temp_ids, genre_column]
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 560
Validation: 120
Test: 120


In [ ]:
import json
import os

os.makedirs("/content/fma/splits", exist_ok=True)

with open("/content/fma/splits/train.json", "w") as f:
    json.dump(train_ids, f)

with open("/content/fma/splits/val.json", "w") as f:
    json.dump(val_ids, f)

with open("/content/fma/splits/test.json", "w") as f:
    json.dump(test_ids, f)

print("Splits saved.")

Splits saved.


# Audio Features

In [ ]:
import librosa
import numpy as np

def extract_audio_features(track_id):
    audio_path = os.path.join(
        "/content/fma/audio",
        f"{int(track_id):06d}.mp3"
    )

    y, sr = librosa.load(
        audio_path,
        sr=CONFIG["sample_rate"],
        mono=True
    )

    segment_length = CONFIG["segment_seconds"] * sr

    # Use the first 30 seconds, padded if necessary
    target_length = 30 * sr

    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]

    features = []

    for start in range(0, len(y), segment_length):
        segment = y[start:start + segment_length]

        if len(segment) < segment_length:
            segment = np.pad(
                segment,
                (0, segment_length - len(segment))
            )

        chroma = librosa.feature.chroma_stft(
            y=segment,
            sr=sr,
            n_chroma=12
        )

        mfcc = librosa.feature.mfcc(
            y=segment,
            sr=sr,
            n_mfcc=13
        )

        chroma_mean = chroma.mean(axis=1)
        mfcc_mean = mfcc.mean(axis=1)

        node_feature = np.concatenate([
            chroma_mean,
            mfcc_mean
        ])

        features.append(node_feature)

    return np.array(features, dtype=np.float32)


test_features = extract_audio_features(selected_ids[0])

print("Feature shape:", test_features.shape)
print("Feature dimension:", test_features.shape[1])

Feature shape: (6, 25)
Feature dimension: 25


# Graphs

In [ ]:
import torch
from torch_geometric.data import Data
from sklearn.metrics.pairwise import cosine_similarity

def build_graph(track_id):
    features = extract_audio_features(track_id)

    num_nodes = features.shape[0]

    edges = set()

    # Temporal adjacency edges
    for i in range(num_nodes - 1):
        edges.add((i, i + 1))
        edges.add((i + 1, i))

    # Similarity edges
    similarity = cosine_similarity(features)

    threshold = 0.75

    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if similarity[i, j] >= threshold:
                edges.add((i, j))
                edges.add((j, i))

    edge_index = torch.tensor(
        list(edges),
        dtype=torch.long
    ).t().contiguous()

    x = torch.tensor(
        features,
        dtype=torch.float
    )

    label = label_to_id[
        tracks.loc[track_id, ("track", "genre_top")]
    ]

    return Data(
        x=x,
        edge_index=edge_index,
        y=torch.tensor([label], dtype=torch.long),
        track_id=torch.tensor([int(track_id)])
    )


# Test graph construction
test_graph = build_graph(selected_ids[0])

print(test_graph)
print("Nodes:", test_graph.num_nodes)
print("Features per node:", test_graph.num_node_features)
print("Edges:", test_graph.num_edges)

Data(x=[6, 25], edge_index=[2, 30], y=[1], track_id=[1])
Nodes: 6
Features per node: 25
Edges: 30


In [ ]:
graph_dir = "/content/fma/graphs"
os.makedirs(graph_dir, exist_ok=True)

all_ids = selected_ids

for i, track_id in enumerate(all_ids):
    output_path = os.path.join(
        graph_dir,
        f"{int(track_id):06d}.pt"
    )

    if not os.path.exists(output_path):
        graph = build_graph(track_id)
        torch.save(graph, output_path)

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(all_ids)}")

print("Graph construction complete.")

Processed 50/800


/usr/local/lib/python3.13/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Processed 100/800
Processed 150/800


/usr/local/lib/python3.13/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/usr/local/lib/python3.13/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Processed 200/800
Processed 250/800
Processed 300/800
Processed 350/800
Processed 400/800
Processed 450/800
Processed 500/800
Processed 550/800
Processed 600/800
Processed 650/800
Processed 700/800
Processed 750/800
Processed 800/800
Graph construction complete.


In [ ]:
from torch.utils.data import Dataset

class GraphDataset(Dataset):
    def __init__(self, track_ids):
        self.track_ids = track_ids

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, idx):
        track_id = self.track_ids[idx]

        path = os.path.join(
            graph_dir,
            f"{int(track_id):06d}.pt"
        )

        return torch.load(
            path,
            weights_only=False
        )


train_graph_dataset = GraphDataset(train_ids)
val_graph_dataset = GraphDataset(val_ids)
test_graph_dataset = GraphDataset(test_ids)

print(len(train_graph_dataset))
print(len(val_graph_dataset))
print(len(test_graph_dataset))

560
120
120


In [ ]:
from torch_geometric.loader import DataLoader

train_graph_loader = DataLoader(
    train_graph_dataset,
    batch_size=16,
    shuffle=True
)

val_graph_loader = DataLoader(
    val_graph_dataset,
    batch_size=16,
    shuffle=False
)

test_graph_loader = DataLoader(
    test_graph_dataset,
    batch_size=16,
    shuffle=False
)

print("Graph DataLoaders ready.")

Graph DataLoaders ready.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool

class GraphSAGEModel(nn.Module):
    def __init__(self, input_dim=25, hidden_dim=64, num_classes=8):
        super().__init__()

        self.conv1 = SAGEConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = SAGEConv(
            hidden_dim,
            hidden_dim
        )

        self.classifier = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        graph_embedding = global_mean_pool(
            x,
            batch
        )

        output = self.classifier(
            graph_embedding
        )

        return output, graph_embedding


gnn_model = GraphSAGEModel()

print(gnn_model)

GraphSAGEModel(
  (conv1): SAGEConv(25, 64, aggr=mean)
  (conv2): SAGEConv(64, 64, aggr=mean)
  (classifier): Linear(in_features=64, out_features=8, bias=True)
)


In [ ]:
device = torch.device("cpu")

gnn_model = gnn_model.to(device)

optimizer = torch.optim.Adam(
    gnn_model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

In [ ]:
def train_gnn_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        output, _ = model(
            batch.x,
            batch.edge_index,
            batch.batch
        )

        loss = criterion(
            output,
            batch.y
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = output.argmax(dim=1)

        correct += (
            predictions == batch.y
        ).sum().item()

        total += batch.y.size(0)

    return total_loss / len(loader), correct / total


def evaluate_gnn(model, loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            output, _ = model(
                batch.x,
                batch.edge_index,
                batch.batch
            )

            predictions = output.argmax(dim=1)

            correct += (
                predictions == batch.y
            ).sum().item()

            total += batch.y.size(0)

    return correct / total

In [ ]:
gnn_history = {
    "train_loss": [],
    "train_acc": [],
    "val_acc": []
}

for epoch in range(10):
    loss, train_acc = train_gnn_epoch(
        gnn_model,
        train_graph_loader
    )

    val_acc = evaluate_gnn(
        gnn_model,
        val_graph_loader
    )

    gnn_history["train_loss"].append(loss)
    gnn_history["train_acc"].append(train_acc)
    gnn_history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch + 1}/10 | "
        f"Loss: {loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

Epoch 1/10 | Loss: 4.1685 | Train Acc: 0.1625 | Val Acc: 0.2167
Epoch 2/10 | Loss: 2.2515 | Train Acc: 0.2589 | Val Acc: 0.2583
Epoch 3/10 | Loss: 1.9345 | Train Acc: 0.3179 | Val Acc: 0.2583
Epoch 4/10 | Loss: 1.8441 | Train Acc: 0.3625 | Val Acc: 0.3250
Epoch 5/10 | Loss: 1.8183 | Train Acc: 0.3607 | Val Acc: 0.3083
Epoch 6/10 | Loss: 1.6659 | Train Acc: 0.4107 | Val Acc: 0.2500
Epoch 7/10 | Loss: 1.6905 | Train Acc: 0.4089 | Val Acc: 0.2583
Epoch 8/10 | Loss: 1.6405 | Train Acc: 0.3946 | Val Acc: 0.3417
Epoch 9/10 | Loss: 1.6007 | Train Acc: 0.4339 | Val Acc: 0.2667
Epoch 10/10 | Loss: 1.6287 | Train Acc: 0.4089 | Val Acc: 0.3000


# GNN Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

def get_gnn_predictions(model, loader):
    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            output, _ = model(
                batch.x,
                batch.edge_index,
                batch.batch
            )

            predictions = output.argmax(dim=1)

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                batch.y.cpu().numpy()
            )

    return np.array(all_labels), np.array(all_predictions)


gnn_true, gnn_pred = get_gnn_predictions(
    gnn_model,
    test_graph_loader
)

print(
    "Accuracy:",
    accuracy_score(gnn_true, gnn_pred)
)

print(
    "Macro-F1:",
    f1_score(
        gnn_true,
        gnn_pred,
        average="macro"
    )
)

print(
    "Micro-F1:",
    f1_score(
        gnn_true,
        gnn_pred,
        average="micro"
    )
)

print(
    classification_report(
        gnn_true,
        gnn_pred,
        target_names=genres
    )
)

Accuracy: 0.36666666666666664
Macro-F1: 0.3330180051965436
Micro-F1: 0.36666666666666664
               precision    recall  f1-score   support

   Electronic       0.50      0.27      0.35        15
 Experimental       0.55      0.40      0.46        15
         Folk       0.34      0.73      0.47        15
      Hip-Hop       0.25      0.40      0.31        15
 Instrumental       0.00      0.00      0.00        15
International       0.45      0.67      0.54        15
          Pop       0.00      0.00      0.00        15
         Rock       0.64      0.47      0.54        15

     accuracy                           0.37       120
    macro avg       0.34      0.37      0.33       120
 weighted avg       0.34      0.37      0.33       120

